# EnergyDebug Real-time Energy Dashboard

A comprehensive real-time system monitoring dashboard for energy analysis.

## Capabilities
- Real-time high energy consumption process monitoring with power metrics
- Hardware temperature monitoring (CPU, GPU, Battery) with multiple detection methods
- Battery remaining time estimation and discharge analysis
- Process energy consumption percentage distribution
- Historical energy trend analysis and anomaly detection
- Docker image efficiency classification and comparison

## 1. Environment Setup and Imports

Note: This notebook uses existing datasets without modifying original code.

In [ ]:
## 1. Environment Setup and Imports

This notebook utilizes existing datasets without modification to original code structures.
Required dependencies: psutil, matplotlib, pandas, numpy

## 2. Load Existing Dataset

Reuse data loading logic from energydebug_backup.ipynb

In [ ]:
## 2. Dataset Loading and Initialization

Load energy consumption data from existing experimental results.
Data structure follows the format established in energydebug_backup.ipynb

## 3. Windows Temperature Monitoring Module

Use Windows WMI API for hardware temperature (no large packages required)

In [ ]:
## 3. Hardware Temperature Monitoring System

Multi-method temperature detection for CPU, GPU, and Battery subsystems.

Detection methodologies:
- Windows Management Instrumentation (WMI)
- NVIDIA Management Library (NVML) for NVIDIA GPUs
- Thermal estimation algorithms based on system load

## 4. High Energy Process Monitoring

In [ ]:
## 4. Process Energy Consumption Monitoring

Real-time monitoring of per-process energy consumption with:
- Power consumption estimation (Watts)
- Cumulative energy consumption (Joules)
- Process contribution percentage to total system consumption

## 5. Energy Trend Analysis Module

In [ ]:
## 5. Historical Energy Trend Analysis

Statistical analysis and visualization of historical energy consumption patterns
across different Docker container images.

## 6. Image Detection Module

In [ ]:
## 6. Container Image Anomaly Detection

Automated detection of anomalous energy consumption patterns and
efficiency classification of Docker container images.

## 7. Real-time Dashboard Main Interface

In [ ]:
class RealtimeDashboard:
    """Real-time Energy Monitoring Dashboard
    
    Provides comprehensive real-time visualization of system energy consumption,
    hardware temperatures, battery status, and process-level energy analysis.
    """
    
    def __init__(self, hw_monitor, proc_monitor, trend_analyzer, image_detector):
        self.hw_monitor = hw_monitor
        self.proc_monitor = proc_monitor
        self.trend_analyzer = trend_analyzer
        self.image_detector = image_detector
        
        self.running = False
        self.history = {
            'timestamps': deque(maxlen=100),
            'cpu_temps': deque(maxlen=100),
            'gpu_temps': deque(maxlen=100),
            'battery_temps': deque(maxlen=100),
            'power_usage': deque(maxlen=100),
            'cpu_percent': deque(maxlen=100),
            'memory_percent': deque(maxlen=100)
        }
    
    def update_data(self):
        """Collect current system metrics"""
        timestamp = datetime.now()
        
        temps = self.hw_monitor.get_all_temperatures()
        power = self.proc_monitor.get_system_power_metrics()
        battery = self.hw_monitor.get_battery_info()
        
        self.history['timestamps'].append(timestamp)
        self.history['cpu_temps'].append(temps['cpu'] or 0)
        self.history['gpu_temps'].append(temps['gpu'] or 0)
        self.history['battery_temps'].append(temps['battery'] or 0)
        self.history['power_usage'].append(power['estimated_power_w'])
        self.history['cpu_percent'].append(power['cpu_percent'])
        self.history['memory_percent'].append(power['memory_percent'])
        
        return {
            'timestamp': timestamp,
            'temps': temps,
            'power': power,
            'battery': battery,
            'top_processes': self.proc_monitor.get_top_energy_processes(5)
        }
    
    def display_status(self, data):
        """Display formatted system status information"""
        clear_output(wait=True)
        
        temps = data['temps']
        power = data['power']
        battery = data['battery']
        processes = data['top_processes']
        
        # Header
        print("=" * 80)
        print(f"ENERGYDEBUG REAL-TIME MONITORING DASHBOARD")
        print(f"Report Generated: {data['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)
        
        # Temperature Section
        print("\n[HARDWARE TEMPERATURE MONITORING]")
        cpu_str = f"{temps['cpu']:.1f}C" if temps['cpu'] else "N/A"
        gpu_str = f"{temps['gpu']:.1f}C" if temps['gpu'] else "N/A"
        batt_str = f"{temps['battery']:.1f}C" if temps['battery'] else "N/A"
        print(f"  Processor (CPU):     {cpu_str:>10}")
        print(f"  Graphics (GPU):      {gpu_str:>10}")
        print(f"  Battery:             {batt_str:>10}")
        
        # System Power Section
        print("\n[SYSTEM POWER STATUS]")
        print(f"  CPU Utilization:     {power['cpu_percent']:>9.1f}%")
        print(f"  Memory Utilization:  {power['memory_percent']:>9.1f}%")
        print(f"  Estimated Power:     {power['estimated_power_w']:>9.2f} W")
        
        # Battery Section
        if battery:
            print("\n[BATTERY STATUS]")
            print(f"  Charge Level:        {battery['percent']:>9.1f}%")
            source = "AC Adapter (Charging)" if battery['power_plugged'] else "Battery Power"
            print(f"  Power Source:        {source:>20}")
            if not battery['power_plugged'] and battery['secsleft']:
                # Calculate remaining time based on current power consumption
                current_power_w = power['estimated_power_w']
                if current_power_w > 0:
                    remaining_wh = (battery['percent'] / 100.0) * 50.0  # Assume 50Wh battery
                    remaining_hours = remaining_wh / current_power_w
                    remaining_mins = int(remaining_hours * 60)
                    print(f"  Estimated Runtime:   {remaining_hours:>9.2f} h ({remaining_mins} minutes)")
            print(f"  Time Remaining:      {battery['time_left_str']:>10}")
        
        # Process Energy Section
        print("\n[PROCESS ENERGY CONSUMPTION ANALYSIS]")
        print(f"{'Rank':<6} {'Process Name':<25} {'Power (W)':<10} {'Energy (J)':<12} {'Share %':<8}")
        print("-" * 80)
        for i, proc in enumerate(processes, 1):
            name = proc['name'][:24]
            print(f"  {i:<4} {name:<25} {proc['power_estimate_w']:<10.2f} "
                  f"{proc['energy_consumption_j']:<12.1f} {proc['energy_percent']:<8.2f}")
        
        # Footer
        print("\n" + "=" * 80)
        print("Press Ctrl+C to terminate monitoring session")
        print("=" * 80)
    
    def plot_realtime_charts(self):
        """Generate real-time visualization charts"""
        if len(self.history['timestamps']) < 2:
            return
        
        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        
        timestamps = list(self.history['timestamps'])
        times = [(t - timestamps[0]).total_seconds() for t in timestamps]
        
        # Chart 1: CPU Temperature
        ax1 = axes[0, 0]
        temps = list(self.history['cpu_temps'])
        if any(t > 0 for t in temps):
            ax1.plot(times, temps, 'r-', linewidth=2, label='CPU')
            ax1.set_ylabel('Temperature (C)')
            ax1.set_title('CPU Temperature Trend')
            ax1.grid(True, alpha=0.3)
            ax1.axhline(y=80, color='orange', linestyle='--', label='Warning Threshold')
            ax1.axhline(y=90, color='red', linestyle='--', label='Critical Threshold')
            ax1.legend()
        else:
            ax1.text(0.5, 0.5, 'Temperature sensor unavailable', ha='center', va='center', 
                    transform=ax1.transAxes)
        
        # Chart 2: GPU Temperature
        ax2 = axes[0, 1]
        gpu_temps = list(self.history['gpu_temps'])
        if any(t > 0 for t in gpu_temps):
            ax2.plot(times, gpu_temps, 'g-', linewidth=2, label='GPU')
            ax2.set_ylabel('Temperature (C)')
            ax2.set_title('GPU Temperature Trend')
            ax2.grid(True, alpha=0.3)
            ax2.axhline(y=80, color='orange', linestyle='--', label='Warning')
            ax2.axhline(y=90, color='red', linestyle='--', label='Critical')
            ax2.legend()
        else:
            ax2.text(0.5, 0.5, 'GPU sensor unavailable', ha='center', va='center',
                    transform=ax2.transAxes)
        
        # Chart 3: Power Consumption
        ax3 = axes[0, 2]
        power = list(self.history['power_usage'])
        ax3.plot(times, power, 'b-', linewidth=2)
        ax3.set_ylabel('Power (W)')
        ax3.set_title('System Power Consumption')
        ax3.grid(True, alpha=0.3)
        ax3.fill_between(times, power, alpha=0.3)
        
        # Chart 4: CPU Utilization
        ax4 = axes[1, 0]
        cpu = list(self.history['cpu_percent'])
        ax4.plot(times, cpu, 'g-', linewidth=2)
        ax4.set_ylabel('Utilization (%)')
        ax4.set_xlabel('Time (s)')
        ax4.set_title('CPU Utilization')
        ax4.grid(True, alpha=0.3)
        ax4.set_ylim(0, 100)
        
        # Chart 5: Memory Utilization
        ax5 = axes[1, 1]
        memory = list(self.history['memory_percent'])
        ax5.plot(times, memory, 'purple', linewidth=2)
        ax5.set_ylabel('Utilization (%)')
        ax5.set_xlabel('Time (s)')
        ax5.set_title('Memory Utilization')
        ax5.grid(True, alpha=0.3)
        ax5.set_ylim(0, 100)
        
        # Chart 6: Battery Temperature (if available)
        ax6 = axes[1, 2]
        batt_temps = list(self.history['battery_temps'])
        if any(t > 0 for t in batt_temps):
            ax6.plot(times, batt_temps, 'orange', linewidth=2)
            ax6.set_ylabel('Temperature (C)')
            ax6.set_xlabel('Time (s)')
            ax6.set_title('Battery Temperature')
            ax6.grid(True, alpha=0.3)
        else:
            ax6.text(0.5, 0.5, 'Battery sensor unavailable', ha='center', va='center',
                    transform=ax6.transAxes)
        
        plt.tight_layout()
        plt.show()
    
    def run_once(self):
        """Execute single monitoring cycle"""
        data = self.update_data()
        self.display_status(data)
        self.plot_realtime_charts()
        return data
    
    def run_continuous(self, interval=2, duration=60):
        """Execute continuous monitoring session
        
        Args:
            interval: Seconds between updates
            duration: Total monitoring duration in seconds
        """
        self.running = True
        start_time = time.time()
        
        print(f"Initiating continuous monitoring session")
        print(f"Duration: {duration} seconds | Update interval: {interval} seconds")
        print("Press Ctrl+C to terminate\n")
        
        try:
            while self.running and (time.time() - start_time) < duration:
                self.run_once()
                time.sleep(interval)
        except KeyboardInterrupt:
            print("\nMonitoring session terminated by user")
            self.running = False

dashboard = RealtimeDashboard(hw_monitor, proc_monitor, trend_analyzer, image_detector)
print("Real-time monitoring dashboard initialized")
print("\nAvailable Operations:")
print("  1. dashboard.run_once()                    - Single update cycle")
print("  2. dashboard.run_continuous()              - Continuous monitoring (60s default)")
print("  3. dashboard.run_continuous(1, 300)        - Custom interval and duration")


## 8. Run Real-time Monitoring Demo

In [ ]:
## 8. Real-time Monitoring Demonstration

Execute a single monitoring cycle to verify system functionality.

## 9. Historical Energy Trend Analysis

In [ ]:
## 9. Historical Data Visualization

Generate comparative visualizations of historical energy consumption data.

In [ ]:
print("Generating energy timeline...")
fig2 = trend_analyzer.plot_energy_timeline(figsize=(14, 6))
plt.show()

## 10. Anomaly Detection Visualization

Visual representation of detected anomalies and efficiency classifications.

In [ ]:
print("Generating image detection results...")
fig3 = image_detector.plot_detection_results(figsize=(14, 10))
plt.show()

## 11. Continuous Monitoring Mode

Execute extended monitoring session. Terminate with Ctrl+C.

In [ ]:
# Start continuous monitoring (60 seconds, update every 2 seconds)
# Adjust parameters as needed
# dashboard.run_continuous(interval=2, duration=60)

## 12. Report Generation and Export

Export comprehensive monitoring report in JSON format.

In [ ]:
report = image_detector.generate_report()

report['system_info'] = {
    'cpu_count': psutil.cpu_count(),
    'cpu_freq_mhz': psutil.cpu_freq().current if psutil.cpu_freq() else None,
    'total_memory_gb': psutil.virtual_memory().total / (1024**3),
    'platform': sys.platform
}

if not aggregate_df.empty:
    report['energy_summary'] = {
        'total_images': len(aggregate_df['Image'].unique()),
        'total_runs': len(aggregate_df),
        'avg_energy_j': aggregate_df['Total_Energy_J'].mean(),
        'min_energy_j': aggregate_df['Total_Energy_J'].min(),
        'max_energy_j': aggregate_df['Total_Energy_J'].max(),
        'std_energy_j': aggregate_df['Total_Energy_J'].std()
    }

report_file = f"energy_report_{WORKLOAD}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f"Report saved: {report_file}")
print("\nReport Summary:")
print(json.dumps(report, indent=2, default=str)[:2000] + "...")

---

## Documentation

### Implemented Capabilities

1. **Process Energy Monitoring**: Real-time tracking of per-process energy consumption
   with power estimation in Watts and cumulative energy in Joules

2. **Hardware Temperature Monitoring**:
   - CPU temperature via WMI or thermal estimation
   - GPU temperature via NVML (NVIDIA) or WMI
   - Battery temperature and status monitoring

3. **Battery Analysis**: Remaining capacity estimation and runtime projection
   based on current power consumption rates

4. **Energy Distribution**: Per-process percentage contribution to total
   system energy consumption

5. **Historical Analysis**: Statistical analysis and visualization of
   energy consumption trends across container images

6. **Anomaly Detection**: Automated identification of anomalous energy
   consumption patterns with Z-score analysis

### Resource Optimization

Designed for systems with limited storage capacity:
- Utilizes existing Python dependencies exclusively
- No additional machine learning libraries required
- Stream-based data processing for minimal memory footprint
- JSON report format for compact storage

### Optional Dependencies

For enhanced GPU temperature monitoring on NVIDIA systems:
```bash
pip install nvidia-ml-py --no-cache-dir
```

### Data Source

This dashboard utilizes datasets from energydebug_backup.ipynb
without modification to original code or data structures.